# 🏦 Notebook 3 — End-to-End Worked Example: Strangling a Banking Monolith

We'll migrate a small legacy banking app to a new service, slice by slice, using
everything from Notebooks 1 and 2 plus two new ideas:

- **Dual-write** for data migration (so both systems see new data during cutover).
- **Per-endpoint progress tracking** with a tiny metrics dashboard.

All in pure Python — no servers, no DB. Just dictionaries pretending to be a bank.

## 🛠️ Setup

```bash
cd 05-microservices/strangler
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This lab uses **only the Python standard library** — no servers, no databases. Every cell is a small simulation you can read end-to-end.


## 1. The legacy bank (don't touch it!)

Four operations: `create_account`, `deposit`, `withdraw`, `balance`.
Data lives in a global dict (pretend it's an ancient Oracle DB).

In [ ]:
LEGACY_DB = {}   # account_id -> balance in cents

class LegacyBank:
    name = "legacy-bank-v1"
    def create_account(self, account_id):
        LEGACY_DB.setdefault(account_id, 0)
        return {"ok": True, "by": self.name}
    def deposit(self, account_id, cents):
        LEGACY_DB[account_id] = LEGACY_DB.get(account_id, 0) + cents
        return {"ok": True, "by": self.name, "balance": LEGACY_DB[account_id]}
    def withdraw(self, account_id, cents):
        if LEGACY_DB.get(account_id, 0) < cents:
            return {"ok": False, "by": self.name, "error": "insufficient_funds"}
        LEGACY_DB[account_id] -= cents
        return {"ok": True, "by": self.name, "balance": LEGACY_DB[account_id]}
    def balance(self, account_id):
        return {"ok": True, "by": self.name, "balance": LEGACY_DB.get(account_id, 0)}


## 2. The new bank (clean rewrite, built one slice at a time)

In [ ]:
NEW_DB = {}

class NewBank:
    name = "new-bank-v2"
    def create_account(self, account_id):
        if account_id in NEW_DB:
            return {"ok": True, "by": self.name, "note": "already_exists"}
        NEW_DB[account_id] = 0
        return {"ok": True, "by": self.name}
    def deposit(self, account_id, cents):
        NEW_DB[account_id] = NEW_DB.get(account_id, 0) + cents
        return {"ok": True, "by": self.name, "balance": NEW_DB[account_id]}
    def withdraw(self, account_id, cents):
        if NEW_DB.get(account_id, 0) < cents:
            return {"ok": False, "by": self.name, "error": "insufficient_funds"}
        NEW_DB[account_id] -= cents
        return {"ok": True, "by": self.name, "balance": NEW_DB[account_id]}
    def balance(self, account_id):
        return {"ok": True, "by": self.name, "balance": NEW_DB.get(account_id, 0)}


## 3. The facade — a richer router

This facade tracks, per operation:
- **% traffic** on the new service,
- whether to **dual-write** (write to both DBs during migration),
- a simple **metrics counter** so we can see what's happening.


In [ ]:
import random, collections, time

class BankingFacade:
    def __init__(self, legacy, new):
        self.legacy = legacy
        self.new = new
        # op -> {"pct_new": 0..100, "dual_write": bool}
        self.plan = collections.defaultdict(lambda: {"pct_new": 0, "dual_write": False})
        self.metrics = collections.Counter()   # (op, target) -> count
        self.latency_ms = collections.defaultdict(list)

    def configure(self, op, pct_new=None, dual_write=None):
        if pct_new is not None:   self.plan[op]["pct_new"] = pct_new
        if dual_write is not None: self.plan[op]["dual_write"] = dual_write

    def _call(self, target, op, *args):
        t0 = time.perf_counter()
        result = getattr(target, op)(*args)
        self.latency_ms[(op, target.name)].append((time.perf_counter() - t0) * 1000)
        self.metrics[(op, target.name)] += 1
        return result

    def handle(self, op, *args):
        cfg = self.plan[op]
        use_new = random.randint(1, 100) <= cfg["pct_new"]
        primary = self.new if use_new else self.legacy
        secondary = self.legacy if use_new else self.new

        result = self._call(primary, op, *args)

        # Dual-write: mirror the write to the other system so its data stays fresh.
        # Only for write ops — never dual-write reads.
        if cfg["dual_write"] and op in {"create_account", "deposit", "withdraw"}:
            try:
                mirrored = self._call(secondary, op, *args)
                # ⚠️ A mirror that returns ok=False is a SILENT divergence — the two
                # systems now disagree and nothing raised. Count it like an error.
                if not mirrored.get("ok", True):
                    self.metrics[("mirror_rejected", secondary.name)] += 1
            except Exception:
                # A mirror failure must never break the primary response...
                self.metrics[("mirror_error", secondary.name)] += 1
                # ...but it must never be invisible either.
        return result

    def report(self):
        print("--- traffic counts ---")
        for (op, target), count in sorted(self.metrics.items()):
            print(f"  {op:16s} {target:16s} {count:5d}")


### The check nobody writes until it's too late: **parity**

Dual-writing is the easy half. The half that bites is that the two systems only agree if
they *started* from the same state — and a system you just built starts empty.

Every real migration therefore needs a **parity check**: read the same key out of both
systems and compare. Run it constantly, and treat any drift as a hard blocker on
shifting reads. Ours is five lines.

In [ ]:
def parity(label):
    """Compare every account balance in both systems. Returns the mismatch count."""
    keys = sorted(set(LEGACY_DB) | set(NEW_DB))
    diffs = [(k, LEGACY_DB.get(k), NEW_DB.get(k))
             for k in keys if LEGACY_DB.get(k) != NEW_DB.get(k)]
    status = "✅ IN PARITY" if not diffs else f"❌ {len(diffs)}/{len(keys)} accounts DISAGREE"
    print(f"  parity after {label}: {status}")
    for k, l, n in diffs:
        print(f"    {k}: legacy={l} new={n}  (drift={None if None in (l, n) else n - l})")
    return len(diffs)

## 4. The migration in 5 phases

We simulate a workload at each phase and watch the routing shift.

In [ ]:
def simulate(facade, n=500):
    """Pretend customers are doing stuff."""
    for _ in range(n):
        aid = random.choice(["A1", "A2", "A3", "A4"])
        op = random.choices(
            ["create_account", "deposit", "withdraw", "balance"],
            weights=[1, 5, 3, 10],
        )[0]
        if op == "create_account":
            facade.handle(op, aid)
        elif op == "balance":
            facade.handle(op, aid)
        else:
            facade.handle(op, aid, random.randint(100, 5000))

random.seed(42)
LEGACY_DB.clear(); NEW_DB.clear()
facade = BankingFacade(LegacyBank(), NewBank())

print("Phase 1 — facade in place, 100% still on legacy (no behavioural change).")
simulate(facade, 300)
facade.report()


In [ ]:
print("\nPhase 2 — SHADOW reads. Call the new service too, compare, throw its answer away.")
print("          (This is the dark launch from Notebook 2. Note what we do NOT do:")
print("           serve those answers to customers. NEW_DB is still empty, so every")
print("           one of them would be wrong.)")

shadow_diffs = 0
def shadow_read(account_id):
    global shadow_diffs
    truth  = facade.legacy.balance(account_id)["balance"]
    shadow = facade.new.balance(account_id)["balance"]
    if truth != shadow:
        shadow_diffs += 1
    return truth                       # the customer only ever sees legacy

for aid in ["A1", "A2", "A3", "A4"] * 25:
    shadow_read(aid)

simulate(facade, 300)
facade.report()
print(f"\nshadow read mismatches: {shadow_diffs}/100")
print("100% wrong — exactly what you would expect from an empty database, and")
print("exactly the signal that says 'do not move reads yet'.")

In [ ]:
print("\nPhase 3 — start DUAL-WRITING deposits. Reads stay on LEGACY.")
facade.configure("deposit", pct_new=0, dual_write=True)   # write both, read legacy
simulate(facade, 300)
facade.report()

print("\nLEGACY_DB:", LEGACY_DB)
print("NEW_DB   :", NEW_DB)
parity("dual-writing deposits")

### 🚨 Dual-write alone does **not** give you parity

Look at that output. The two systems disagree — often by thousands of cents. Two reasons,
and both are structural, not bugs in the demo:

1. **NEW_DB started empty.** It never saw the deposits and withdrawals that happened in
   Phases 1 and 2, so it is missing all history from before dual-write was switched on.
2. **We only mirrored `deposit`.** `withdraw` still goes to legacy alone, so every
   withdrawal widens the gap.

If we had flipped `balance` reads to the new service here — which is exactly what a
plausible-looking phase plan would do — customers would now be shown a **wrong balance**,
with no error anywhere to tell us. Dual-write makes the two systems *converge going
forward*; it cannot invent the past.

**The rule: mirror every write, then backfill, then verify parity, and only then move
reads.**

In [ ]:
# --- Step 1: mirror EVERY write op, not just deposits ---
for op in ("create_account", "deposit", "withdraw"):
    facade.configure(op, pct_new=0, dual_write=True)

# --- Step 2: BACKFILL. Copy the historical state legacy already has.
# In production this is a bulk export/import (or CDC replay) that runs while
# dual-write keeps both systems moving. Idempotent, restartable, and it *must*
# happen while writes are already being mirrored — otherwise you race the
# writes you copied.
for account_id, cents in LEGACY_DB.items():
    NEW_DB[account_id] = cents
print("backfill complete: copied", len(LEGACY_DB), "accounts legacy → new")

# --- Step 3: verify ---
simulate(facade, 300)          # keep real traffic flowing during the check
facade.report()
print()
drift = parity("backfill + full dual-write")
print()
print("Only now is it safe to move reads. In production you would run this parity")
print("check on a schedule for days, and alert on ANY non-zero drift, before")
print("touching the read path.")

In [ ]:
print("\nPhase 4a — parity is green, so NOW we canary reads: 100% of `balance` to new.")
facade.configure("balance", pct_new=100)
simulate(facade, 300)
parity("read cutover")

print("\nPhase 4b — flip the writes too. Keep dual-writing BACK to legacy so that")
print("            the kill switch remains real: legacy stays a valid fallback.")
facade.configure("deposit",        pct_new=100, dual_write=True)
facade.configure("withdraw",       pct_new=100, dual_write=True)
facade.configure("create_account", pct_new=100, dual_write=True)
simulate(facade, 300)
facade.report()
print()
parity("write cutover (legacy still mirrored)")

In [ ]:
print("\nPhase 5 — retirement: confident, turn dual-write off. Legacy stops receiving traffic.")
for op in ["create_account", "deposit", "withdraw", "balance"]:
    facade.configure(op, pct_new=100, dual_write=False)

# Reset counters so this phase's report shows ONLY the phase-5 traffic.
facade.metrics.clear()
facade.latency_ms.clear()

simulate(facade, 300)
facade.report()
print("\nNotice: no `legacy-bank-v1` rows. Legacy received 0 calls in phase 5.")
print()
drift = parity("dual-write OFF")
print()
print("⚠️  From this moment the two systems drift apart on every single write, and")
print("    the kill switch stops being a rollback — it becomes a data-loss event.")
print("    Turning dual-write off is the real point of no return in a migration,")
print("    not the traffic flip. Do it last, deliberately, after days of green parity.")


## 5. Looking at the latency log

In [ ]:
def p(name, vals):
    if not vals: return
    vals = sorted(vals)
    p50 = vals[len(vals)//2]
    p95 = vals[int(len(vals)*0.95)]
    print(f"  {name:40s} n={len(vals):4d}  p50={p50:.3f}ms  p95={p95:.3f}ms")

print("Latency by (operation, target):")
for key, vals in sorted(facade.latency_ms.items()):
    p(f"{key[0]} @ {key[1]}", vals)


## 6. Lessons from this worked example

1. **Phase 1 is a no-op deploy** — putting the facade in place with 0% migration is its
   own scary deploy. Do it on its own.
2. **Reads before writes** — but *only after parity*. `balance` is read-only and safe to
   move first; moving it before the data matched would have shown customers wrong
   balances with no error to catch it.
3. **Dual-write ≠ parity.** Mirroring writes makes the systems converge *from now on*.
   The new system still starts empty, so you also need a **backfill**, and you need it to
   run *while* dual-write is already on, or you race the writes you just copied.
4. **Parity is a gate, not a report.** Automate it, alert on any drift, and refuse to
   shift the read path while it's red. A silently wrong balance is far worse than a 500.
5. **Watch for mirror rejections, not just exceptions.** A mirrored `withdraw` that comes
   back `insufficient_funds` is a divergence that raised nothing — count it.
6. **The facade owns the migration state.** Nothing in legacy or new needs to know a
   migration is happening. That's the whole point.
7. **Retirement is deliberate.** Turning dual-write off — not the traffic flip — is when
   rollback stops being free.

## 7. 🌍 Variations you'll see in the wild

- **Event interception** — instead of HTTP routing, intercept at the *event bus*
  (e.g. Kafka). The new service subscribes to the same events; when ready, the
  old consumer is turned off. Used heavily in CQRS / event-sourced systems.
- **Branch by abstraction** — introduce an interface inside the legacy codebase
  and slowly reimplement it. Good when you *can* modify legacy code but want
  to avoid a big-bang branch.
- **Change data capture (CDC)** — instead of dual-writing from the app, use a
  tool like Debezium to stream every legacy DB change into the new DB. The app
  stays unchanged; the DB is strangled.
- **Strangler at the UI layer** — mount the new app on a sub-path (`/v2/*`) and
  migrate pages one at a time. LinkedIn and Etsy have written about this.

All of them are the same idea: **one routing layer, one tiny slice at a time,
always reversible.**


## 8. 📚 Further reading

- Martin Fowler — *StranglerFigApplication* (2004) — the original essay.
- Sam Newman — *Monolith to Microservices* (O'Reilly, 2019) — the canonical book.
- GitHub's *Scientist* library — [github/scientist](https://github.com/github/scientist) — production dark-launch tool.
- Shopify engineering blog — "Deconstructing the Monolith".
- Stripe engineering — "Online migrations at scale".

### 🎯 You now know
- Why big-bang rewrites fail, and how the strangler fig avoids that.
- The three pieces: legacy, new, facade.
- Canary routing, dark launch, kill switches.
- Dual-write for data migration.
- The five phases of a real migration — including retirement.
